# Project T: Real vs. Fake Faces Demo
**Group Name:** IowaGPT  
**Members:** Amit Boodhoo, Diego Liogon, Eva Singh, Kate Meyer  

**Description:** This notebook loads our pre-trained ResNet-18 model and evaluates it on 10 independent test images we collected ourselves.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import os

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 1. Recreate the exact architecture we trained
model = models.resnet18(weights=None) 
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)     # 2 classes: Real and Fake

# 2. Load the saved weights from our training notebook
model_path = 'resnet18_fake_faces.pt'
model.load_state_dict(torch.load(model_path, map_location=device))

# 3. CRITICAL: Set the model to evaluation mode
model = model.to(device)
model.eval()
print("Model loaded successfully and set to evaluation mode.")

In [ ]:
# Use the exact same resizing and normalization as the training data
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

classes = ['Fake', 'Real'] # Update this order if your PyTorch loaded them differently!

def predict_image(image_path):
    """Loads an image, runs it through the model, and plots the result."""
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device) 
    
    with torch.no_grad():
        output = model(input_tensor)
        _, predicted_idx = torch.max(output, 1)
        confidence = torch.nn.functional.softmax(output, dim=1)[0][predicted_idx].item()
        
    predicted_label = classes[predicted_idx.item()]
    
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"Prediction: {predicted_label} ({confidence*100:.1f}%)")
    plt.show()

In [ ]:
# Point this to the folder containing your 10 custom images
custom_test_dir = '10_Custom_Test_Images'

# Loop through all images in the folder and predict
for filename in os.listdir(custom_test_dir):
    if filename.endswith(('.png', '.jpg', '.jpeg')):
        image_path = os.path.join(custom_test_dir, filename)
        print(f"Testing image: {filename}")
        predict_image(image_path)